# പാഠം 11 - ഏജന്റ്-ടു-ഏജന്റ് (A2A) പ്രോട്ടോക്കോൾ


## സജ്ജീകരണം


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## A2A പ്രോട്ടോക്കോൾ എന്താണ്?

**Agent-to-Agent (A2A) protocol** ഒരു തുറന്ന സ്റ്റാൻഡാർഡാണ്, ഇത് AI ഏജന്റുകൾക്ക് പരസ്പരം ആശയവിനിമയം നടത്താനും, പരസ്പരം കണ്ടെത്താനും, സഹകരിക്കാനും അനുവദിക്കുന്നു — അവ വ്യത്യസ്ത ഫ്രേയിംവർക്കുകളിൽ നിർമ്മിച്ചിട്ടുണ്ടെങ്കിലും അല്ലെങ്കിൽ വ്യത്യസ്ത സേവനങ്ങളിലൂടെ ഹോസ്റ്റ് ചെയ്തിട്ടുണ്ടെങ്കിലും പോലും.

Key concepts:

- **Discovery** – ഏജന്റുകൾ അവരുടെ കഴിവുകൾ വിവരിക്കുന്ന *ഏജന്റ് കാർഡ്* പ്രസിദ്ധീകരിക്കുന്നു, ഇതിലൂടെ മറ്റു ഏജന്റുകൾ (അഥവാ ഓർക്കസ്ട്രേറ്ററുകൾ) ഒരു ജോലിക്ക് അനുയോജ്യനായ വിദഗ്ധനെ കണ്ടെത്താൻ എളുപ്പമാകും.
- **Message Passing** – ഏജന്റുകൾ പൊതുവായ ഒരു പ്രോട്ടോക്കോൾ വഴി ഘടനയുള്ള സന്ദേശങ്ങൾ കൈമാറുന്നു, അതിനാൽ ഒരു ഏജന്റിന്റെ അഭ്യർത്ഥന മറ്റൊരു ഏജന്റ് അതിന്റെ ആന്തരിക നടപ്പാക്കലിനെ ആശ്രയിക്കാതെ മനസ്സിലാക്കി പാലിക്കാനാകും.
- **Task Lifecycle** – A2A *സമർപ്പിച്ചത്*, *പ്രവർത്തനത്തിൽ*, *സമാപ്തം*, *പരാജയപ്പെട്ടത്* പോലുള്ള നിലകൾ നിർവചിക്കുന്നു, ഇതിലൂടെ ഓർക്കസ്ട്രേറ്ററിന് നിയുക്ത ടാസ്കിന്റെ പുരോഗതി എങ്ങിനെയാണ് എന്നതിൽ പൂർണ്ണ ദൃശ്യത ലഭിക്കുന്നു.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents into a workflow where each agent contributes its expertise and passes results to the next.


## പ്രത്യേകമായ യാത്രാ ഏജന്റുകൾ സൃഷ്ടിക്കൽ


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## പ്രവർത്തനപ്രവാഹം വഴി ബഹുഎജന്റ് സഹകരണം

ഞങ്ങൾ A2A സന്ദേശ കൈമാറ്റത്തെ പ്രതിഫലിപ്പിക്കുന്ന ക്രമാനുസൃത പ്രവർത്തനപ്രവാഹമായി ആ മൂന്ന് ഏജന്റുകളെയും ബന്ധിപ്പിക്കുന്നു:

1. **CurrencyExchangeAgent** ഉപയോക്തൃ അഭ്യർത്ഥന സ്വീകരിച്ച് നാണയ മാർഗ്ഗനിർദ്ദേശം നൽകുന്നു.
2. **ActivityPlannerAgent** സമ്പുഷ്ടമായ സാന്ദർഭ്യം സ്വീകരിച്ച് പ്രവർത്തന ശിപാർശകൾ ചേർക്കുന്നു.
3. **TravelManagerAgent** ഇരുവരും നൽകുന്ന ഇൻപുട്ടുകളെ സംയോജിപ്പിച്ച് അന്തിമ യാത്രാസംഗ്രഹം രൂപപ്പെടുത്തുന്നു.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## പ്രൊഡക്ഷനിലുള്ള A2A മനസ്സിലാക്കൽ

ഒരു പ്രൊഡക്ഷൻ പരിസരത്തിൽ A2A പ്രോട്ടോക്കോൾ ശക്തമായ ക്രോസ്-സർവീസ് സീനാരിയോകൾ സജ്ജമാക്കുന്നു:

| ശേഷി | വിവരണം |
|---|---|
| **ഫ്രെയിംവർക്കുകൾ ഇടയിലുള്ള അന്തർഓപ്പറബിലിറ്റി** | ഒരു ഫ്രെയിംവർക്കിലുള്ള ആയോജകൻ മറ്റേതെങ്കിലും A2A-കമ്പ്ലയന്റ് ഫ്രെയിംവർക്കിൽ നിർമ്മിച്ച ഏജന്റിനോട് ടാസ്കുകൾ നിയോഗിക്കാൻ കഴിയും, ഇത് സംഘടനകളിടയിലെ യഥാർത്ഥ അന്തർഓപ്പറബിലിത്ത്വം സജ്ജമാക്കുന്നു. |
| **സേവന അതിരുകൾ** | ഏജന്റുകൾ വേർതിരിച്ച മൈക്രോസെർവിസുകളിൽ, ക്ലൗഡ് റീജിയനുകളിൽ, അല്ലെങ്കിൽ വ്യത്യസ്ത സംഘടനകളിൽ നിലനിൽക്കുകയും അതോടൊപ്പം തടസ്സമില്ലാതെ സഹകരിക്കയും ചെയ്യാം. |
| **ഡൈനാമിക് കണ്ടെത്തൽ** | ഒരു ഓർക്കസ്ട്രേറ്റർ റൺടൈമിൽ Agent Card രജിസ്ട്രിയെ ക്വറി ചെയ്ത് നിശ്ചിത ഉപ-ടാസ്കിനായുള്ള ഏറ്റവും അനുയോജ്യമായ വിദഗ്ധനെ കണ്ടെത്താൻ കഴിയും. |
| **സ്റ്റ്രീമിംഗ് & പുഷ് നോട്ടിഫിക്കേഷനുകൾ** | A2A തത്സമയ പുരോഗതി അപ്‌ഡേറ്റുകൾക്കായി Server-Sent Events (SSE) നെയും ദൈർഘ്യമേറിയ ടാസ്കുകൾക്കായി പുഷ് നോട്ടിഫിക്കേഷനുകളെയും പിന്തുണയ്ക്കുന്നു. |

മുകളിൽ നാം നിർമിച്ച വർക്ക്‌ഫ്ലോ ഈ മാതൃകയുടെ ലളിതമാക്കിയ, ഇൻ-പ്രോസ്സസ് പതിപ്പാണ്. യഥാർത്ഥ വിന്യാസത്തിൽ ഓരോ ഏജന്റും ഒരു HTTP endpoint പുറത്തുവിടുകയും, ഒരു Agent Card പ്രസിദ്ധീകരിക്കുകയും, A2A JSON-RPC പ്രോട്ടോക്കോൾ വഴിയേ_commUNICate ചെയ്യുകയും ചെയ്യും.


## സംഗ്രഹം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. **A2A പ്രോട്ടോക്കോൾ എന്താണ്** — എജന്റ്-ടു-എജന്റ് കണ്ടെത്തൽ, സന്ദേശവിനിമയം,
   ടാസ്‌ക് മാനേജ്മെന്റ്.
2. **പ്രತ್ಯേക ഏജന്റുകൾ എങ്ങനെ സൃഷ്ടിക്കാമെന്ന്** — ഒരു നാണയവിനിമയ ഏജന്റ്, ഒരു ആക്ടിവിറ്റി പ്ലാനർ ഏജന്റ്,
   കൂടാതെ ഒരു ട്രാവൽ മാനേജർ ഓർക്കസ്ട്രേറ്റർ.
3. **ഏജന്റുകൾ ഒരു വർക്ക്‌ഫ്ലോയിൽ എങ്ങനെ ബന്ധിപ്പിക്കാമെന്ന്** — `WorkflowBuilder` ഉപയോഗിച്ച് അനുക്രമമായ
   സന്ദേശ കൈമാറ്റം ഏജന്റുകൾക്കിടയിൽ മോഡൽ ചെയ്യാൻ.
4. **A2A പ്രൊഡക്ഷനിൽ എങ്ങനെ പ്രവർത്തിക്കുന്നു** — ക്രോസ്-ഫ്രെയിംവർക്ക്, ക്രോസ്-സർവീസ് സഹകരണം സാധ്യമാക്കുന്നു
   ഡൈനാമിക് കണ്ടെത്തലും സ്ട്രീമിംഗ് അപ്‌ഡേറ്റുകളും ഉപയോഗിച്ച്.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
അസ്വീകരണ കുറിപ്പ്:
ഈ രേഖ AI വിവർത്തന സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്ക് ശ്രമിച്ചിരിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് വിവർത്തനങ്ങളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉൾപ്പെട്ടിരിക്കാമെന്ന് ദയവായി അറിയിക്കുക. അസൽഭാഷയിലെ മൂല പ്രമാണം അധികാരപരമായ ഉറവിടമായി കരുതപ്പെടണം. നിർണായക माहितीക്കായി പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം നിർദ്ദേശിക്കുന്നു. ഈ വിവർത്തനത്തിന്റെ ഉപയോഗത്തിൽ നിന്നുയർന്ന任何 തെറ്റിദ്ധാരണകളിലേക്കോ തെറ്റായ വ്യാഖ്യാനങ്ങളിലേക്കോ ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
